# Robustness Sweeps Across Noise and Subsampling

This notebook explores how the shipped `v0.6` stack behaves under simple robustness perturbations.

It uses:

- `add_gaussian_noise(...)`
- `subsample_time(...)`
- `subsample_x(...)`
- `split_batch_train_heldout(...)`
- `evaluate_discovery_recovery(...)`
- `summarize_recovery_grid(...)`

The recovery metric here is **not** PDE-truth recovery. It compares backend-native sparse terms from each perturbed fit against a clean baseline backend-native fit on the same primary state feature.


In [ ]:
import importlib.util

if importlib.util.find_spec("pysindy") is None:
    raise RuntimeError("Install pdelie[downstream] or pdelie[test] to run this notebook.")

import numpy as np

from pdelie import GeneratorFamily
from pdelie.data import (
    add_gaussian_noise,
    generate_heat_1d_field_batch,
    split_batch_train_heldout,
    subsample_time,
    subsample_x,
)
from pdelie.discovery import (
    build_translation_canonical_discovery_inputs,
    evaluate_discovery_recovery,
    fit_pysindy_discovery,
    summarize_recovery_grid,
)


In [ ]:
def make_known_translation_generator() -> GeneratorFamily:
    basis_spec = {
        "variables": ["t", "x", "u"],
        "component_names": ["xi"],
        "basis_terms": [
            {"label": "1", "powers": [0, 0, 0]},
            {"label": "t", "powers": [1, 0, 0]},
            {"label": "x", "powers": [0, 1, 0]},
            {"label": "u", "powers": [0, 0, 1]},
        ],
        "component_ordering": ["xi"],
        "term_ordering": ["1", "t", "x", "u"],
        "layout": "component_major",
    }
    return GeneratorFamily(
        parameterization="polynomial_translation_affine",
        coefficients=np.array([[1.0, 0.0, 0.0, 0.0]], dtype=float),
        basis_spec=basis_spec,
        normalization="l2_unit",
        diagnostics={},
    )


def extract_primary_terms(fit_result, feature_name):
    return dict(fit_result["equation_terms"].get(feature_name, {}))


def run_condition(base_field, generator, *, std_fraction, noise_seed, time_stride, x_stride):
    noisy = add_gaussian_noise(base_field, std_fraction=std_fraction, seed=noise_seed)
    time_subsampled = subsample_time(noisy, stride=time_stride)
    x_subsampled = subsample_x(time_subsampled, stride=x_stride)
    train_field, heldout_field = split_batch_train_heldout(x_subsampled, train_size=2, seed=7000)

    discovery_inputs = build_translation_canonical_discovery_inputs(train_field, generator_family=generator)
    fit_result = fit_pysindy_discovery(
        discovery_inputs["trajectories"],
        discovery_inputs["time_values"],
        discovery_inputs["feature_names"],
    )
    primary_feature = discovery_inputs["feature_names"][0]
    return {
        "train_field": train_field,
        "heldout_field": heldout_field,
        "inputs": discovery_inputs,
        "fit": fit_result,
        "primary_feature": primary_feature,
        "primary_terms": extract_primary_terms(fit_result, primary_feature),
    }


In [ ]:
base_field = generate_heat_1d_field_batch(batch_size=4, num_times=33, num_points=64, seed=200)
generator = make_known_translation_generator()

baseline_run = run_condition(
    base_field,
    generator,
    std_fraction=0.0,
    noise_seed=9000,
    time_stride=1,
    x_stride=1,
)
baseline_terms = baseline_run["primary_terms"]
baseline_feature = baseline_run["primary_feature"]

print("baseline primary feature:", baseline_feature)
print("baseline fit status:", baseline_run["fit"]["status"])
print("baseline nonzero backend terms:", len(baseline_terms))


## Sweep conditions

We repeat each condition across a few noise seeds so `summarize_recovery_grid(...)` has multiple records per condition.


In [ ]:
records = []
fit_rows = []

for std_fraction in [0.0, 1e-3, 1e-2, 5e-2]:
    for time_stride in [1, 2, 4]:
        for x_stride in [1, 2, 4]:
            for noise_seed in [9100, 9101, 9102]:
                run = run_condition(
                    base_field,
                    generator,
                    std_fraction=std_fraction,
                    noise_seed=noise_seed,
                    time_stride=time_stride,
                    x_stride=x_stride,
                )
                recovery = evaluate_discovery_recovery(baseline_terms, run["primary_terms"])
                records.append(
                    {
                        "conditions": {
                            "noise_std_fraction": std_fraction,
                            "time_stride": time_stride,
                            "x_stride": x_stride,
                        },
                        "recovery": recovery,
                    }
                )
                fit_rows.append(
                    {
                        "noise_std_fraction": std_fraction,
                        "time_stride": time_stride,
                        "x_stride": x_stride,
                        "noise_seed": noise_seed,
                        "fit_status": run["fit"]["status"],
                        "primary_nonzero_terms": len(run["primary_terms"]),
                        "classification": recovery["classification"],
                    }
                )

fit_rows[:5]


In [ ]:
summary_rows = summarize_recovery_grid(records)
summary_rows[:8]


## A few easy comparisons

These are simple slices through the grouped summaries. They are often enough to spot where the current `v0.6` stack begins to look structurally brittle.


In [ ]:
exact_rows = [row for row in summary_rows if row["exact_count"] > 0]
failed_rows = [row for row in summary_rows if row["failed_count"] > 0]

print("num grouped conditions:", len(summary_rows))
print("conditions with at least one exact recovery against baseline support:", len(exact_rows))
print("conditions with at least one failed recovery against baseline support:", len(failed_rows))

print("\nRepresentative grouped rows:")
for row in summary_rows[:5]:
    print(row)
